In [16]:
from dotenv import load_dotenv
load_dotenv()

from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from minsearch import Index
from rag_helper import RAGBase
from openai import OpenAI

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

import json

openai_client = OpenAI()

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Question 1: How many lesson pages
How many lesson pages are in the dataset?

In [5]:
print(f"Parsed {len(documents)} documents.")

Parsed 72 documents.


## Question 2: Indexing and searching
Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [6]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = 'How does the agentic loop keep calling the model until it stops?'
search_results = index.search(
    question,
    #boost_dict={"question": 2.0},
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

## Question 3: RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).


In [7]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)
answer, usage = assistant.rag(question)
print(f"Answer: {answer}\nUsage: {usage}")

Answer: It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, it runs the tools, appends the tool outputs to the message history, and loops again.
- If there are no function calls, it breaks and returns the final answer.

So the stop condition is: **no function calls in the model’s response**.
Usage: ResponseUsage(input_tokens=7036, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7129)


## Question 4: Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* 295
* 1100
* 4500

In [8]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks.")

Created 295 chunks.


## Question 5: RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer

In [9]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

question = 'How does the agentic loop keep calling the model until it stops?'
search_results = index.search(
    question,
    num_results=5
)

search_results

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)
answer, usage = assistant.rag(question)
print(f"Answer: {answer}\nUsage: {usage}")

Answer: It keeps calling the model inside a `while True` loop. After each model response, the code checks whether any `function_call` items were returned:

- If there are function calls, it runs them, appends the results to `messages`, and loops again.
- If there are no function calls on that turn, it breaks out of the loop.

So the stopping condition is: **no function calls this turn means the agent is done**.
Usage: ResponseUsage(input_tokens=2221, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2316)


## Question 6: Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20

In [21]:
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
""".strip()

# Save reference to chunks index
chunks_index = index

def search(query: str) -> dict[str, str]:
    """
    Search the document chunks for information matching the given query.
    """
    return chunks_index.search(
        query,
        num_results=5
    )

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print(f"Query cost: {result.cost}")

print(f"All messages: {result.all_messages}")

-> Response received


-> Response received


Query cost: CostInfo(input_cost=Decimal('0.006105'), output_cost=Decimal('0.002394'), total_cost=Decimal('0.008499'))
All messages: [EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the\nsearch tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG differences retrieval generation loop"}', call_id='call_9rTYzbKA4QG2Vfwlcbcw628w', name='search', type='function_call', id='fc_09feb9ac38f62327006a3049a17f288199a56261626a366d5d', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop what is it in course notes"}', call_id='call_C39Z8eA8Dk5w4ntks1ahRZz6', name='search', type='function_call', id='fc_09feb9ac38f62327006a3049a17f388199867d8cb826

## OpenAI Agents SDK (Swarm)

In [23]:
# Define the search tool for OpenAI function calling
def search_documents(query: str) -> dict[str, str]:
    """Search the document chunks for information matching the given query."""
    return chunks_index.search(query, num_results=5)

# Define the tool schema for OpenAI
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the document chunks for information matching the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Create an OpenAI client
client = OpenAI()

# Agentic loop
messages = [
    {
        "role": "system",
        "content": instructions
    },
    {
        "role": "user",
        "content": "How does the agentic loop work, and how is it different from plain RAG?"
    }
]

# Track search calls and token usage
search_call_count = 0
total_input_tokens = 0
total_output_tokens = 0

# Run the agent loop
while True:
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages,
        tools=tools
    )
    
    # Extract message and tool calls from response
    message = response.choices[0].message
    
    # Track tokens
    total_input_tokens += response.usage.prompt_tokens
    total_output_tokens += response.usage.completion_tokens
    
    # Add assistant response to messages
    messages.append({
        "role": "assistant",
        "content": message.content,
        "tool_calls": message.tool_calls
    })
    
    # Check if we should stop
    if response.choices[0].finish_reason == "end_turn":
        print("Assistant response:")
        print(message.content)
        break
    
    # If there are tool calls, process them
    if message.tool_calls:
        for tool_call in message.tool_calls:
            if tool_call.function.name == "search_documents":
                search_call_count += 1
                args = json.loads(tool_call.function.arguments)
                result = search_documents(args["query"])
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
    else:
        # No more tool calls, we're done
        print("Assistant response:")
        print(message.content)
        break

print(f"\n--- Agent Metrics ---")
print(f"Search tool calls: {search_call_count}")
print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Total tokens: {total_input_tokens + total_output_tokens}")

Assistant response:
The agentic loop is basically a **repeat-until-done tool-use loop**:

1. You send the model a user question plus instructions.
2. The model decides whether it needs a tool call, like search.
3. If it does, your code runs the tool.
4. You send the tool result back to the model.
5. The model can then:
   - answer directly, or
   - ask for another tool call
6. Repeat until the model returns a final answer with **no more tool calls**.

A simple version looks like:

```python
while True:
    response = llm(messages)
    if response has tool call:
        run tool
        append tool result to messages
    else:
        break
```

### How it differs from plain RAG

**Plain RAG** is usually a fixed pipeline:

```python
search_results = search(question)
prompt = build_prompt(question, search_results)
answer = llm(prompt)
```

That means:

- search happens **once**
- the query is usually the exact user question
- the model does **not** get to revise the search
- if retrieval